# 02. Olist 다중 테이블 Base Table 설계와 JOIN 검증

## 목적

이 노트북은 Olist 이커머스 프로젝트에서 만든 분석용 base table 구조를 검증하는 문서다.

Olist 데이터는 주문, 상품, 결제, 리뷰, 고객, 판매자 테이블의 기준 단위가 서로 다르다.  
따라서 분석 전에 각 테이블의 grain을 이해하고, 분석 질문에 맞는 base table을 분리해야 한다.

이 노트북의 핵심은 pandas로 단순히 JOIN을 할 수 있다는 것이 아니라, **JOIN으로 인해 발생할 수 있는 중복 집계 위험을 이해하고 분석 목적에 맞는 테이블을 설계할 수 있음**을 보여주는 것이다.

## 보여주는 역량

- SQLite DB 구조 확인
- 테이블별 grain 정의
- naive JOIN 위험 검증
- 주문 단위 사전 집계 테이블 검증
- 주문 단위 base table 검증
- 상품 단위 base table 검증
- 분석 질문별 적절한 base table 선택 기준 정리

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/02_olist")
OUTPUT_DIR = Path("../outputs/02_olist")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

db_path = DATA_DIR / "olist_ecommerce.db"
conn = sqlite3.connect(db_path)

## 1. DB 테이블 구조 확인

먼저 SQLite DB에 어떤 테이블이 들어 있는지 확인한다.

In [ ]:
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
    conn
)
tables

In [ ]:
table_row_counts = []
for table in tables["name"]:
    row_count = pd.read_sql(f'SELECT COUNT(*) AS row_count FROM "{table}"', conn)["row_count"].iloc[0]
    table_row_counts.append({"table": table, "row_count": row_count})

table_row_counts = pd.DataFrame(table_row_counts)
table_row_counts.to_csv(OUTPUT_DIR / "02_table_row_counts.csv", index=False, encoding="utf-8-sig")
table_row_counts

In [ ]:
schema_rows = []
for table in tables["name"]:
    info = pd.read_sql(f'PRAGMA table_info("{table}")', conn)
    schema_rows.append({
        "table": table,
        "column_count": len(info),
        "columns": ", ".join(info["name"].tolist())
    })

table_schema_summary = pd.DataFrame(schema_rows)
table_schema_summary.to_csv(OUTPUT_DIR / "02_table_schema_summary.csv", index=False, encoding="utf-8-sig")
table_schema_summary

## 2. 테이블별 grain 확인

각 테이블의 행 수와 `order_id` 또는 주요 key의 고유 수를 비교한다.  
이 단계는 어떤 테이블이 주문 단위인지, 상품 단위인지, 결제수단 단위인지 확인하기 위한 과정이다.

In [ ]:
grain_definitions = {
    "orders": "주문 1건",
    "customers": "customer_id 1개",
    "order_items": "주문-상품 1행",
    "order_payments": "주문-결제수단 1행",
    "order_reviews": "리뷰 1행",
    "products": "상품 1개",
    "sellers": "판매자 1명",
    "orders_enriched": "주문 1건 + 배송 파생변수",
    "payment_by_order": "주문 1건 기준 결제 사전 집계",
    "review_by_order": "주문 1건 기준 리뷰 사전 집계",
    "order_base_delivered": "배송 완료 주문 1건",
    "order_item_base_delivered": "배송 완료 주문상품 1행",
}

key_col_map = {
    "orders": "order_id",
    "customers": "customer_id",
    "order_items": "order_id",
    "order_payments": "order_id",
    "order_reviews": "order_id",
    "products": "product_id",
    "sellers": "seller_id",
    "orders_enriched": "order_id",
    "payment_by_order": "order_id",
    "review_by_order": "order_id",
    "order_base_delivered": "order_id",
    "order_item_base_delivered": "order_id",
}

grain_rows = []
for table, grain in grain_definitions.items():
    if table not in tables["name"].values:
        continue

    key = key_col_map[table]
    row_count = pd.read_sql(f'SELECT COUNT(*) AS n FROM "{table}"', conn)["n"].iloc[0]
    unique_key_count = pd.read_sql(f'SELECT COUNT(DISTINCT "{key}") AS n FROM "{table}"', conn)["n"].iloc[0]

    grain_rows.append({
        "table": table,
        "grain_definition": grain,
        "key_column_for_check": key,
        "row_count": row_count,
        "unique_key_count": unique_key_count,
        "duplicate_key_count": row_count - unique_key_count,
        "is_key_unique": row_count == unique_key_count
    })

table_grain_summary = pd.DataFrame(grain_rows)
table_grain_summary.to_csv(OUTPUT_DIR / "02_table_grain_summary.csv", index=False, encoding="utf-8-sig")
table_grain_summary

## 3. naive JOIN 위험 검증

아래 JOIN은 실제 분석에 사용하기 위한 것이 아니라, 왜 원본 테이블을 그대로 한 번에 JOIN하면 위험한지 보여주기 위한 예시다.

주문 1건에 여러 상품, 여러 결제 행, 리뷰가 연결되면 JOIN 후 행 수가 늘어나고 결제금액이나 리뷰 점수가 중복 집계될 수 있다.

In [ ]:
naive_join_risk = pd.read_sql(
    '''
    SELECT
        (SELECT COUNT(*) FROM orders) AS orders_rows,
        (SELECT COUNT(*) FROM order_items) AS item_rows,
        (SELECT COUNT(*) FROM order_payments) AS payment_rows,
        (SELECT COUNT(*) FROM order_reviews) AS review_rows,
        COUNT(*) AS naive_join_rows,
        COUNT(DISTINCT o.order_id) AS distinct_orders_after_join,
        SUM(oi.price) AS sum_item_price_after_join,
        SUM(op.payment_value) AS sum_payment_value_after_join,
        AVG(r.review_score) AS avg_review_score_after_join
    FROM orders o
    LEFT JOIN order_items oi ON o.order_id = oi.order_id
    LEFT JOIN order_payments op ON o.order_id = op.order_id
    LEFT JOIN order_reviews r ON o.order_id = r.order_id
    ''',
    conn
)

original_reference = pd.read_sql(
    '''
    SELECT
        (SELECT SUM(price) FROM order_items) AS original_item_price_sum,
        (SELECT SUM(payment_value) FROM order_payments) AS original_payment_value_sum,
        (SELECT AVG(review_score) FROM order_reviews) AS original_review_score_avg
    ''',
    conn
)

naive_join_risk = pd.concat([naive_join_risk, original_reference], axis=1)
naive_join_risk.to_csv(OUTPUT_DIR / "02_naive_join_risk_summary.csv", index=False, encoding="utf-8-sig")
naive_join_risk

### 해석

JOIN 후 행 수가 증가하는 것은 그 자체로 오류가 아니다.  
문제는 이 결과를 주문 단위 KPI 계산에 그대로 사용하면 결제금액, 상품가격, 리뷰점수가 중복 집계될 수 있다는 점이다.

따라서 주문 단위 KPI를 만들 때는 결제와 리뷰를 먼저 주문 단위로 사전 집계해야 한다.

## 4. 사전 집계 테이블 검증

`payment_by_order`와 `review_by_order`는 원본 결제/리뷰 테이블을 주문 단위로 사전 집계한 테이블이다.

이 테이블들이 `order_id` 기준으로 고유한지 확인한다.

In [ ]:
pre_aggregation_summary = pd.DataFrame([
    {
        "source_table": "order_payments",
        "pre_aggregated_table": "payment_by_order",
        "source_rows": pd.read_sql("SELECT COUNT(*) AS n FROM order_payments", conn)["n"].iloc[0],
        "source_unique_order_id": pd.read_sql("SELECT COUNT(DISTINCT order_id) AS n FROM order_payments", conn)["n"].iloc[0],
        "pre_aggregated_rows": pd.read_sql("SELECT COUNT(*) AS n FROM payment_by_order", conn)["n"].iloc[0],
        "pre_aggregated_unique_order_id": pd.read_sql("SELECT COUNT(DISTINCT order_id) AS n FROM payment_by_order", conn)["n"].iloc[0],
        "message": "결제 원본은 주문 1건에 여러 행이 있을 수 있어 주문 단위로 사전 집계"
    },
    {
        "source_table": "order_reviews",
        "pre_aggregated_table": "review_by_order",
        "source_rows": pd.read_sql("SELECT COUNT(*) AS n FROM order_reviews", conn)["n"].iloc[0],
        "source_unique_order_id": pd.read_sql("SELECT COUNT(DISTINCT order_id) AS n FROM order_reviews", conn)["n"].iloc[0],
        "pre_aggregated_rows": pd.read_sql("SELECT COUNT(*) AS n FROM review_by_order", conn)["n"].iloc[0],
        "pre_aggregated_unique_order_id": pd.read_sql("SELECT COUNT(DISTINCT order_id) AS n FROM review_by_order", conn)["n"].iloc[0],
        "message": "리뷰 원본은 주문별 리뷰 결측/중복 가능성이 있어 주문 단위로 사전 집계"
    }
])

pre_aggregation_summary.to_csv(OUTPUT_DIR / "02_pre_aggregation_summary.csv", index=False, encoding="utf-8-sig")
pre_aggregation_summary

## 5. 주문 단위 base table 검증

`order_base_delivered`는 배송 완료 주문 1건을 기준으로 만든 분석용 테이블이다.

월별 주문 수, 매출, 객단가, 배송 지연율, 리뷰 점수 분석에 사용한다.

In [ ]:
order_base_validation = pd.read_sql(
    '''
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT order_id) AS unique_order_id,
        COUNT(*) - COUNT(DISTINCT order_id) AS duplicate_order_id_count,
        COUNT(DISTINCT customer_id) AS unique_customer_id,
        COUNT(DISTINCT customer_unique_id) AS unique_customer_unique_id,
        MIN(purchase_date) AS min_purchase_date,
        MAX(purchase_date) AS max_purchase_date,
        SUM(CASE WHEN order_status != 'delivered' THEN 1 ELSE 0 END) AS non_delivered_rows,
        SUM(CASE WHEN payment_value IS NULL THEN 1 ELSE 0 END) AS missing_payment_value_rows,
        SUM(CASE WHEN review_score IS NULL THEN 1 ELSE 0 END) AS missing_review_score_rows,
        AVG(delivery_days) AS avg_delivery_days,
        AVG(delay_days) AS avg_delay_days,
        AVG(is_delayed) AS delay_rate
    FROM order_base_delivered
    ''',
    conn
)

order_base_validation.to_csv(OUTPUT_DIR / "02_order_base_validation.csv", index=False, encoding="utf-8-sig")
order_base_validation

In [ ]:
order_base_monthly_kpi = pd.read_sql(
    '''
    SELECT
        purchase_month,
        COUNT(DISTINCT order_id) AS order_count,
        SUM(payment_value) AS revenue,
        AVG(payment_value) AS avg_order_value,
        AVG(delivery_days) AS avg_delivery_days,
        AVG(is_delayed) AS delay_rate,
        AVG(review_score) AS avg_review_score
    FROM order_base_delivered
    GROUP BY purchase_month
    ORDER BY purchase_month
    ''',
    conn
)

order_base_monthly_kpi.to_csv(OUTPUT_DIR / "02_order_base_monthly_kpi_sample.csv", index=False, encoding="utf-8-sig")
order_base_monthly_kpi.head()

## 6. 상품 단위 base table 검증

`order_item_base_delivered`는 배송 완료 주문상품 1행 기준 테이블이다.

카테고리별 매출, 상품 가격, 판매자 지역, 상품 단위 리뷰 점수 분석에 사용한다.  
이 테이블에서는 `order_id`가 중복되는 것이 정상이다. 주문 1건에 여러 상품이 포함될 수 있기 때문이다.

In [ ]:
item_base_validation = pd.read_sql(
    '''
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT order_id) AS unique_order_id,
        COUNT(*) - COUNT(DISTINCT order_id) AS duplicate_order_id_count,
        COUNT(*) - COUNT(DISTINCT order_id || '-' || order_item_id) AS duplicate_order_item_key_count,
        COUNT(DISTINCT product_id) AS unique_product_id,
        COUNT(DISTINCT seller_id) AS unique_seller_id,
        SUM(price) AS item_price_sum,
        AVG(price) AS avg_item_price,
        AVG(review_score) AS avg_review_score
    FROM order_item_base_delivered
    ''',
    conn
)

item_base_validation.to_csv(OUTPUT_DIR / "02_item_base_validation.csv", index=False, encoding="utf-8-sig")
item_base_validation

In [ ]:
item_base_category_revenue = pd.read_sql(
    '''
    SELECT
        product_category,
        COUNT(*) AS order_item_count,
        COUNT(DISTINCT order_id) AS order_count,
        SUM(price) AS item_revenue,
        AVG(price) AS avg_item_price,
        AVG(review_score) AS avg_review_score
    FROM order_item_base_delivered
    GROUP BY product_category
    ORDER BY item_revenue DESC
    LIMIT 20
    ''',
    conn
)

item_base_category_revenue.to_csv(OUTPUT_DIR / "02_item_base_category_revenue_sample.csv", index=False, encoding="utf-8-sig")
item_base_category_revenue.head()

## 7. 분석 질문별 base table 선택 기준

분석 질문에 따라 사용해야 하는 base table이 다르다.

주문 단위 KPI는 `order_base_delivered`, 상품/카테고리 분석은 `order_item_base_delivered`를 사용한다.

In [ ]:
analysis_question_table_mapping = pd.DataFrame([
    {
        "analysis_question": "월별 주문 수는 어떻게 변화했는가?",
        "recommended_table": "order_base_delivered",
        "grain": "배송 완료 주문 1건",
        "reason": "주문 수는 order_id 기준으로 중복 없이 집계해야 함"
    },
    {
        "analysis_question": "월별 매출과 객단가는 어떻게 변화했는가?",
        "recommended_table": "order_base_delivered",
        "grain": "배송 완료 주문 1건",
        "reason": "결제금액은 payment_by_order로 주문 단위 사전 집계되어 있음"
    },
    {
        "analysis_question": "배송 지연율은 지역별로 어떻게 다른가?",
        "recommended_table": "order_base_delivered",
        "grain": "배송 완료 주문 1건",
        "reason": "배송일과 지연 여부는 주문 단위 지표"
    },
    {
        "analysis_question": "배송 지연 구간별 리뷰 점수는 어떻게 다른가?",
        "recommended_table": "order_base_delivered",
        "grain": "배송 완료 주문 1건",
        "reason": "리뷰 점수는 review_by_order로 주문 단위 사전 집계되어 있음"
    },
    {
        "analysis_question": "카테고리별 매출 기여도는 어떻게 다른가?",
        "recommended_table": "order_item_base_delivered",
        "grain": "배송 완료 주문상품 1행",
        "reason": "상품 카테고리와 price는 주문상품 행 기준 지표"
    },
    {
        "analysis_question": "판매자 지역별 상품 매출은 어떻게 다른가?",
        "recommended_table": "order_item_base_delivered",
        "grain": "배송 완료 주문상품 1행",
        "reason": "seller_id와 seller_state는 상품 행에 연결되는 분석 기준"
    },
    {
        "analysis_question": "상품 가격 분포는 어떻게 나타나는가?",
        "recommended_table": "order_item_base_delivered",
        "grain": "배송 완료 주문상품 1행",
        "reason": "상품 가격 price는 item 단위 컬럼"
    },
])

analysis_question_table_mapping.to_csv(OUTPUT_DIR / "02_analysis_question_table_mapping.csv", index=False, encoding="utf-8-sig")
analysis_question_table_mapping

## 최종 요약

이 노트북의 핵심은 pandas나 SQL로 JOIN을 실행할 수 있다는 것이 아니다.

핵심은 다음과 같다.

1. 테이블별 기준 단위가 다르면 JOIN 후 지표가 왜곡될 수 있다.
2. 결제와 리뷰는 주문 단위로 사전 집계한 뒤 결합해야 한다.
3. 월별 KPI, 배송 지연율, 리뷰 점수는 주문 단위 base table에서 분석해야 한다.
4. 카테고리별 매출, 상품 가격, 판매자 분석은 상품 단위 base table에서 분석해야 한다.
5. 분석 질문에 맞는 grain을 먼저 정하고 base table을 선택해야 한다.

따라서 `order_base_delivered`와 `order_item_base_delivered`를 분리한 것은 단순 편의가 아니라, 중복 집계 위험을 줄이고 분석 지표의 해석 가능성을 높이기 위한 설계 결정이다.

In [ ]:
conn.close()